In [1]:
#!pip install -U altair vegafusion vegafusion-python-embed "vl-convert-python>=1.6.0"
import pandas as pd
import altair as alt

alt.data_transformers.enable("vegafusion")
#alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('vegafusion')

In [3]:
# 1. Load the dataset
df = pd.read_csv('uber_dataset.csv')

# 2. Replace nulls
df = df.replace("null", pd.NA)

# 3. Extracting Time/Date
df['Time'] = pd.to_datetime(df['Time'], format="%H:%M:%S", errors='coerce')
df['Hour'] = df['Time'].dt.hour

df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
df["Weekday"] = df["Date"].dt.day_name()

# 4. Convert numbers
df['Booking Value'] = pd.to_numeric(df['Booking Value'], errors="coerce")
df['Ride Distance'] = pd.to_numeric(df['Ride Distance'], errors="coerce")

# 5. Create a subset of top 20 pickup locations by number of rides
top_20_pickups = df["Pickup Location"].value_counts().nlargest(20).index
df_pickups_top20 = df[df["Pickup Location"].isin(top_20_pickups)]

# 6. Create a list of vehicle types
vehicle_options = ["All"] + sorted(df["Vehicle Type"].dropna().unique().tolist())

print('Data is loaded and prepared successfully')

Data is loaded and prepared successfully


In [ ]:
# 1. Interaction
pickup_selection = alt.selection_point(fields=["Pickup Location"])
drop_selection = alt.selection_point(fields=["Drop Location"])

# Create a dropdown list for vehicle filtering (supporting T1 and T5)
vehicle_dropdown = alt.param(
    name="VehicleTypeFilter",
    bind=alt.binding_select(options=vehicle_options, name="Vehicle Type: "),
    value="All"
)
vehicle_filter = "VehicleTypeFilter == 'All' || datum['Vehicle Type'] == VehicleTypeFilter"

# 2. Charts
# View 1: Pickup Locations
pickup_chart = (
    alt.Chart(df_pickups_top20)
    .transform_filter(vehicle_filter) # Filter locations by Vehicle Type
    .mark_bar()
    .encode(
        x=alt.X("count():Q", title="Number of Rides"),
        y=alt.Y('Pickup Location:N', sort='-x', title="Pickup Location"),
        # Booking Status colouring with a legend (supporting T4)
        color=alt.Color(
            'Booking Status:N',
            legend=alt.Legend(orient="left"),
            title='Booking Status'
        ),
        # Selection indication by changing opacity
        opacity=alt.condition(pickup_selection, alt.value(1), alt.value(0.6)),
        tooltip=[
            alt.Tooltip("Pickup Location:N", title="Pickup Location"),
            alt.Tooltip("count():Q", title="Number of Rides"),
            alt.Tooltip("Booking Status:N", title="Booking Status"),
        ]
    )
    .properties(
        width=350,
        height=400,
        title="Pickup Locations (Top 20)"
    )
    .add_params(pickup_selection)
)

# View 2: Drop Locations
drop_chart = (
    alt.Chart(df)
    .transform_filter(vehicle_filter) # Filter locations by Vehicle Type
    .transform_filter(pickup_selection) # Filter locations by selected Pickup Locations
    .transform_aggregate(
        ride_count='count()',
        groupby=['Drop Location']
    )
    # Rank the locations by total number of rides in descending order
    .transform_window(
        row_number='row_number()',
        sort=[alt.SortField('ride_count', order='descending')]
    )
    .transform_filter("datum.row_number <= 20") # Keep only the top 20 destinations
    .mark_bar()
    .encode(
        x=alt.X("ride_count:Q", title="Number of Rides"),
        y=alt.Y("Drop Location:N", sort="-x"),
        # Selection indication by changing opacity
        opacity=alt.condition(drop_selection, alt.value(1), alt.value(0.6)),
        tooltip=[
            alt.Tooltip("Drop Location:N", title="Drop Location"),
            alt.Tooltip("ride_count:Q", title="Number of Rides"),
        ]
    )
    .properties(
        width=350,
        height=400,
        title="Drop Locations (Top 20)"
    )
    .add_params(drop_selection)
)

# View 3: Heatmap of Ride Demand by Hour and Weekday (supporting T2)
heatmap = (
    alt.Chart(df)
    .transform_filter(vehicle_filter) # Filter map by Vehicle Type
    .transform_filter(pickup_selection) # Filter map by selected Pickup Locations
    .transform_filter(drop_selection) # Filter map by selected Drop Locations
    .mark_rect()
    .encode(
        x=alt.X("Hour:O", title="Hour of Day"),
        y=alt.Y(
            "Weekday:N",
            title="Day of Week",
            sort=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        ),
        # Heatmap colouring with a legend
        color=alt.Color(
                "count():Q",
                scale=alt.Scale(scheme="yellowgreenblue"),
                title="Ride Demand",
                legend=alt.Legend(orient="right")
        ),
        tooltip=[
            alt.Tooltip("Weekday:N", title="Day"),
            alt.Tooltip("Hour:O", title="Hour"),
            alt.Tooltip("count():Q", title="Ride Demand"),
            # Data about average booking value and ride distance including all filters (supporting T3)
            alt.Tooltip("mean(Booking Value):Q", title="Avg Booking Value (₹)", format=".2f"),
            alt.Tooltip("mean(Ride Distance):Q", title="Avg Ride Distance (km)", format=".2f")
        ]
    )
    .properties(
        width=500,
        height=250,
        title="Ride Demand by Hour and Weekday"
    )
)

# 3. Combine both charts and a heatmap into a whole system
system_c = (
    ((pickup_chart | drop_chart) & heatmap)
    .resolve_legend(color="independent")
    .add_params(vehicle_dropdown)
)

# Export to HTML
html_filename = 'System_C.html'
system_c.save(html_filename)

system_c